In [3]:
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import signal
import glob
from datetime import datetime, timedelta
import argparse

In [4]:
def read_miniseed_file(file_path, sampling_rate):
    """
    Read a mini seed file using ObsPy (you need to install it with pip install obspy)
    """
    try:
        from obspy import read
        st = read(file_path)
        # Merge traces if there are multiple traces in the stream
        st.merge(fill_value=0)
        # Get the trace data as a numpy array
        trace_data = st[0].data
        # Get the start time of the trace
        start_time = st[0].stats.starttime.datetime
        return trace_data, start_time
    except ImportError:
        print("ObsPy is not installed. Please install it with 'pip install obspy'")
        # For demonstration, return random data
        print(f"Returning random data for {file_path}")
        # Assuming 24 hours of data at the given sampling rate
        data_length = int(24 * 3600 * sampling_rate)
        random_data = np.random.randn(data_length)
        file_name = os.path.basename(file_path)
        # Try to extract date from filename (assuming format has YYYYMMDD somewhere)
        try:
            date_str = ''.join(c for c in file_name if c.isdigit())[:8]
            start_time = datetime.strptime(date_str, '%Y%m%d')
        except:
            start_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        return random_data, start_time

def compute_stft_and_dominant_freq(data, start_time, sampling_rate, window_length, overlap):
    """
    Compute Short Time Fourier Transform and extract dominant frequencies
    
    Parameters:
    - data: input time series data
    - start_time: start time of the data
    - sampling_rate: sampling rate in Hz
    - window_length: STFT window length in seconds
    - overlap: overlap between windows in seconds
    
    Returns:
    - dominant_freqs: list of dominant frequencies
    - segment_times: list of times corresponding to each segment
    - frequencies: array of frequency bins
    - stft_magnitude: magnitude of STFT (for potential spectrogram plotting)
    """
    # Convert window and overlap from seconds to samples
    nperseg = int(window_length * sampling_rate)
    noverlap = int(overlap * sampling_rate)
    
    # Ensure nperseg is even for FFT efficiency
    if nperseg % 2 != 0:
        nperseg += 1
    
    # Compute STFT
    frequencies, times, Zxx = signal.stft(data, fs=sampling_rate, 
                                         window='hann', 
                                         nperseg=nperseg, 
                                         noverlap=noverlap, 
                                         detrend='constant', 
                                         return_onesided=True, 
                                         boundary=None, 
                                         padded=True)
    
    # Calculate magnitude (power) of STFT
    stft_magnitude = np.abs(Zxx)**2
    
    # Find dominant frequency at each time point
    dominant_freqs = []
    for t in range(stft_magnitude.shape[1]):
        # Get index of maximum power at this time
        max_idx = np.argmax(stft_magnitude[:, t])
        # Get corresponding frequency
        dominant_freq = frequencies[max_idx]
        dominant_freqs.append(dominant_freq)
    
    # Calculate segment times
    segment_times = []
    for t in times:
        # Convert time offset (in seconds) to datetime
        segment_time = start_time + timedelta(seconds=t)
        segment_times.append(segment_time)
    
    return dominant_freqs, segment_times, frequencies, stft_magnitude

def analyze_all_files(input_dir, output_dir, sampling_rate, window_length, overlap):
    """
    Process all mini seed files in the input directory
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all mini seed files (assuming they have .mseed extension)
    # Modify the extension pattern based on your actual file extensions
    all_files = all_files = glob.glob(os.path.join(input_dir, '*.004'))
    
    if not all_files:
        print(f"No mini seed files found in {input_dir}")
        return
    
    # Sort files by name to ensure chronological order
    all_files.sort()
    
    # Process each file
    all_dominant_freqs = []
    all_segment_times = []
    all_stft_results = []  # For storing full STFT results if needed
    
    for file_path in all_files:
        print(f"Processing {file_path}")
        try:
            # Read the data
            data, start_time = read_miniseed_file(file_path, sampling_rate)
            
            # Compute STFT and extract dominant frequencies
            dominant_freqs, segment_times, frequencies, stft_magnitude = compute_stft_and_dominant_freq(
                data, start_time, sampling_rate, window_length, overlap
            )
            
            # Append to our results
            all_dominant_freqs.extend(dominant_freqs)
            all_segment_times.extend(segment_times)
            
            # Store STFT results for this file
            file_result = {
                'file_name': os.path.basename(file_path),
                'start_time': start_time,
                'frequencies': frequencies,
                'stft_magnitude': stft_magnitude
            }
            all_stft_results.append(file_result)
            
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
    
    return all_dominant_freqs, all_segment_times, all_stft_results

def plot_results(dominant_freqs, segment_times, output_dir):
    """
    Plot frequency vs time and save the figure
    """
    plt.figure(figsize=(16, 8))
    
    # Plot dominant frequency over time
    plt.subplot(1, 1, 1)
    plt.plot(segment_times, dominant_freqs)
    plt.xlabel('Time')
    plt.ylabel('Dominant Frequency (Hz)')
    plt.title('Dominant Frequency Analysis (STFT)')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_dir, 'frequency_dominance_plot.png'), dpi=300)
    plt.close()

def plot_spectrogram(file_result, output_dir):
    """
    Plot spectrogram for a file
    """
    frequencies = file_result['frequencies']
    stft_magnitude = file_result['stft_magnitude']
    file_name = file_result['file_name']
    start_time = file_result['start_time']
    
    plt.figure(figsize=(16, 8))
    
    # Calculate time axis values
    times = np.arange(stft_magnitude.shape[1]) * (stft_magnitude.shape[1] / 86400)  # Assuming 24 hour recording
    time_labels = [start_time + timedelta(seconds=t * 86400 / stft_magnitude.shape[1]) for t in range(0, stft_magnitude.shape[1], stft_magnitude.shape[1]//10)]
    
    # Plot spectrogram (log scale for better visualization)
    plt.pcolormesh(range(stft_magnitude.shape[1]), frequencies, 10 * np.log10(stft_magnitude + 1e-10), shading='gouraud')
    plt.colorbar(label='Power/Frequency (dB/Hz)')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time')
    plt.title(f'Spectrogram: {file_name}')
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_dir, f'spectrogram_{file_name}.png'), dpi=300)
    plt.close()

def save_results(dominant_freqs, segment_times, stft_results, output_dir):
    """
    Save results as a numpy file
    """
    # Convert datetime objects to strings for saving
    time_strings = [t.strftime('%Y-%m-%d %H:%M:%S') for t in segment_times]
    
    # Save to numpy file
    np.savez(os.path.join(output_dir, 'frequency_dominance_results.npz'),
             dominant_frequencies=np.array(dominant_freqs),
             segment_times=np.array(time_strings))
    
    # Save full STFT results (optional - might be large)
    # We'll save just the first file's full STFT as an example
    if stft_results and len(stft_results) > 0:
        first_file = stft_results[0]
        np.savez(os.path.join(output_dir, f'stft_data_{first_file["file_name"]}.npz'),
                frequencies=first_file['frequencies'],
                stft_magnitude=first_file['stft_magnitude'],
                file_name=first_file['file_name'],
                start_time=np.array(first_file['start_time'].strftime('%Y-%m-%d %H:%M:%S')))
    
    print(f"Results saved to {os.path.join(output_dir, 'frequency_dominance_results.npz')}")

In [5]:
def main():
    input_dir = r"D:\Temp"
    output_dir = r"D:\Projects\magma-rsam"
    sampling_rate = 1000.0
    
    # Process all files
    dominant_freqs, segment_times, stft_results = analyze_all_files(
        input_dir,
        output_dir,
        sampling_rate,
        600,
        300
    )
    
    if dominant_freqs:
        # Plot results
        plot_results(dominant_freqs, segment_times, output_dir)
        
        # Plot spectrogram for each file (if you have lots of files, you might want to limit this)
        if stft_results and len(stft_results) > 0:
            for file_result in stft_results[:5]:  # Limit to first 5 files to avoid too many plots
                plot_spectrogram(file_result, output_dir)
        
        # Save results
        save_results(dominant_freqs, segment_times, stft_results, output_dir)
        
        print("Analysis complete!")
    else:
        print("No data processed. Check input directory and file formats.")

In [6]:
main()

Processing D:\Temp\VG.BANG.00.EHZ.D.2024.004
Results saved to D:\Projects\magma-rsam\frequency_dominance_results.npz
Analysis complete!


In [8]:
tes = np.load(r'D:\Projects\magma-rsam\frequency_dominance_results.npz')

In [11]:
len(tes['dominant_frequencies'])

28